# Extract tag + area lookups (for Explore tab)

Pulls the two small lookup tables the app's Explore tab needs:
- `mb_tag.parquet` — tag id → name + ref_count (required)
- `mb_area.parquet` — area id → name + type (optional, enables the country filter)

Output → `data/raw/`. Fast (both tables are small).

In [ ]:
import os
import duckdb
from dotenv import load_dotenv

load_dotenv()
PG = (f"host={os.getenv('PG_HOST','localhost')} port={os.getenv('PG_PORT','5432')} "
      f"dbname={os.getenv('PG_DBNAME','musicbrainz_db')} "
      f"user={os.getenv('PG_USER','musicbrainz')} password={os.getenv('PG_PASSWORD','musicbrainz')}")

con = duckdb.connect()
con.execute("INSTALL postgres; LOAD postgres;")
con.execute(f"ATTACH '{PG}' AS mb (TYPE postgres, READ_ONLY);")

os.makedirs('./data/raw', exist_ok=True)

# --- mb_tag (required) ---
tag = con.execute("""
    SELECT id, name, ref_count
    FROM mb.musicbrainz.tag
""").fetch_df()
tag.to_parquet('./data/raw/mb_tag.parquet', compression='zstd')
print(f'mb_tag:  {len(tag):,} tags  ->  data/raw/mb_tag.parquet')

# --- mb_area (optional, for country filter) ---
area = con.execute("""
    SELECT id, name, type
    FROM mb.musicbrainz.area
""").fetch_df()
area.to_parquet('./data/raw/mb_area.parquet', compression='zstd')
print(f'mb_area: {len(area):,} areas  ->  data/raw/mb_area.parquet')

con.close()

# Quick peek at top tags
print('\nTop 15 tags by ref_count:')
print(tag.nlargest(15, 'ref_count')[['name', 'ref_count']].to_string(index=False))